In [1]:
import random
import hashlib

def modinv(a, m):
    # find b such that a * b ≡ 1 (mod m)
    for i in range(1, m):
        if (a * i) % m == 1:
            return i
    return None

p = 23
g = 5
q = p - 1  # group order (just using p-1 here)
x = 6      # secret key
y = pow(g, x, p)  # public key

challenge_space_size = 4  # e in {0,1,2,3}

random.seed(1)  # to make results repeatable

# 1. Honest Interactive Run
print("--- Honest Interactive Run ---")

# prover chooses random r
r = random.randint(1, q - 1)
t = pow(g, r, p)  # commitment

# verifier chooses random e
e = random.randint(0, challenge_space_size - 1)

# prover computes s = r + e*x (mod q)
s = (r + e * x) % q

# verifier checks g^s ?= t * y^e (mod p)
left = pow(g, s, p)
right = (t * pow(y, e, p)) % p

print("Prover commitment t =", t)
print("Verifier challenge e =", e)
print("Response s =", s)
if left == right:
    print("Verification: Passed")
else:
    print("Verification: Failed")

# 2. One Cheating Attempt
print("--- Cheating Attempt ---")

# cheating prover does not know x
# he guesses the challenge in advance
e_guess = random.randint(0, challenge_space_size - 1)
s_cheat = random.randint(0, q - 1)

y_to_e_guess = pow(y, e_guess, p)
inv_y_to_e_guess = modinv(y_to_e_guess, p)
t_cheat = (pow(g, s_cheat, p) * inv_y_to_e_guess) % p

# verifier picks real challenge (unknown to cheater)
e_real = random.randint(0, challenge_space_size - 1)

left2 = pow(g, s_cheat, p)
right2 = (t_cheat * pow(y, e_real, p)) % p

print("Cheating prover commitment t =", t_cheat)
print("Verifier challenge e =", e_real)
print("Cheater’s guessed e =", e_guess)
print("Response s =", s_cheat)
if left2 == right2:
    print("Verification: Passed")
else:
    print("Verification: Failed")

# 3. Fiat-Shamir (Non-Interactive)
print("--- Fiat-Shamir (Non-Interactive) ---")

message = "Test message"

# Prover side (non-interactive)
r_fs = random.randint(1, q - 1)
t_fs = pow(g, r_fs, p)

# challenge is now a hash of (t || message)
h = hashlib.sha256()
h.update(str(t_fs).encode())
h.update(message.encode())
digest = h.digest()
e_fs = int.from_bytes(digest, "big") % challenge_space_size

s_fs = (r_fs + e_fs * x) % q

# Verifier side
# recompute challenge from t and message
h2 = hashlib.sha256()
h2.update(str(t_fs).encode())
h2.update(message.encode())
digest2 = h2.digest()
e_fs_check = int.from_bytes(digest2, "big") % challenge_space_size

left3 = pow(g, s_fs, p)
right3 = (t_fs * pow(y, e_fs_check, p)) % p

print("Commitment t =", t_fs)
print("Hash-based challenge =", e_fs)
print("Response s =", s_fs)
if e_fs == e_fs_check and left3 == right3:
    print("Verification: Passed")
else:
    print("Verification: Failed")

# 4. Cheating Probability Experiment
print("--- Cheating Probability Experiment ---")

runs = 100
success = 0

for i in range(runs):
    # cheating prover again
    e_guess2 = random.randint(0, challenge_space_size - 1)
    s_cheat2 = random.randint(0, q - 1)

    y_to_e_guess2 = pow(y, e_guess2, p)
    inv_y_to_e_guess2 = modinv(y_to_e_guess2, p)
    t_cheat2 = (pow(g, s_cheat2, p) * inv_y_to_e_guess2) % p

    e_real2 = random.randint(0, challenge_space_size - 1)

    left_loop = pow(g, s_cheat2, p)
    right_loop = (t_cheat2 * pow(y, e_real2, p)) % p

    if left_loop == right_loop:
        success += 1

cheat_rate = success / runs
print("Cheating success rate =", cheat_rate, "(after", runs, "runs)")

--- Honest Interactive Run ---
Prover commitment t = 20
Verifier challenge e = 0
Response s = 5
Verification: Passed
--- Cheating Attempt ---
Cheating prover commitment t = 21
Verifier challenge e = 3
Cheater’s guessed e = 2
Response s = 3
Verification: Failed
--- Fiat-Shamir (Non-Interactive) ---
Commitment t = 19
Hash-based challenge = 2
Response s = 5
Verification: Passed
--- Cheating Probability Experiment ---
Cheating success rate = 0.22 (after 100 runs)
